In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------
PROJECT_DIR = Path("/Users/vedikabaradwaj/Documents/pdi_pensions")
DATA_DIR    = PROJECT_DIR / "regression_analysis" / "analytic samples" / "recruitment"

OUTPUT_DIR  = Path.home() / "Documents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_PAIRS = [
    (2008, 2009),
    (2009, 2010),
    (2010, 2011),
    (2011, 2012),
    (2012, 2013),
    (2013, 2014),
    (2014, 2015),
    (2015, 2016),
    (2016, 2017),
]

FILE_PATTERN = "analytic_sample_{yy0}{yy1}_recruitment.csv"

DV            = "joined_state_local"
TREATMENT_VAR = "illinois"
PAIR_VAR      = "pair"

WEIGHT_CANDIDATES = [
    "weight_t", "wgt_t", "wtfinl_t", "finalwgt_t",
    "weight", "wgt", "wtfinl", "finalwgt",
]

# When log(earnwke_t) is included as a control, use earnwt_t (the CPS earnings
# weight) rather than the demographic weight_t. earnwt_t is assigned to
# observations with valid earnings records, correctly reflecting the probability
# of selection for earnings measurement.
EARNWT_CANDIDATES = ["earnwt_t", "earnwt"]

COV_TYPE = "HC1"

EXCLUDED_STATE_FIPS  = {42}
EXCLUDED_STATE_NAMES = {"pa"}

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists?", DATA_DIR.exists())
if DATA_DIR.exists():
    print("CSV files found:")
    print([p.name for p in sorted(DATA_DIR.glob("*.csv"))])

# -------------------------------------------------------------------
# CONTROL VARIABLES
# -------------------------------------------------------------------
CONTINUOUS_CONTROL_CANDIDATES = [
    "age_t",
    "I(age_t**2)",
    "np.log(earnwke_t)",
]

CATEGORICAL_CONTROL_CANDIDATES = [
    "C(sex_t)",
    "C(race_t)",
    "C(ethnic_t)",
    "C(ethnicity_t)",
    "C(hispan_t)",
    "C(hispanic_t)",
    "C(hisp_t)",
    "C(grade92_t)",
    "C(educ_t)",
    "C(docc00_t)",
    "C(docc80_t)",
    "C(occ2010_t)",
    "C(occ_t)",
    "C(ind02_t)",
    "C(naics2_t)",
    "C(ind_t)",
    "C(unionmme_t)",
    "C(unioncov_t)",
]

ALL_CONTROL_CANDIDATES = CONTINUOUS_CONTROL_CANDIDATES + CATEGORICAL_CONTROL_CANDIDATES


def extract_needed_columns(terms):
    cols = set()
    for term in terms:
        for m in re.findall(r"C\(([^)]+)\)", term):
            cols.add(m.strip())
        for m in re.findall(r"np\.log\(([^)]+)\)", term):
            cols.add(m.strip())
        for expr in re.findall(r"I\(([^)]+)\)", term):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
        if not term.startswith(("C(", "I(", "np.log(")):
            for v in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", term):
                if v not in {"I", "C", "np", "log"}:
                    cols.add(v)
    return sorted(cols)


def is_categorical_term(term):
    return term.startswith("C(")


def categorical_column_from_term(term):
    m = re.match(r"C\(([^)]+)\)", term)
    return m.group(1).strip() if m else None


def choose_one_present(terms, priority):
    present = [t for t in priority if t in terms]
    if len(present) <= 1:
        return terms
    keep = present[0]
    return [t for t in terms if (t not in priority or t == keep)]


def available_terms(df, candidate_terms):
    terms = []
    for term in candidate_terms:
        raw_cols = extract_needed_columns([term])
        if all(c in df.columns for c in raw_cols):
            terms.append(term)
    terms = choose_one_present(terms, ["C(docc00_t)", "C(docc80_t)", "C(occ2010_t)", "C(occ_t)"])
    terms = choose_one_present(terms, ["C(ind02_t)", "C(naics2_t)", "C(ind_t)"])
    terms = choose_one_present(terms, ["C(hispan_t)", "C(hispanic_t)", "C(hisp_t)", "C(ethnic_t)", "C(ethnicity_t)"])
    return terms


def build_formula(dv, controls):
    rhs_terms = [TREATMENT_VAR] + controls
    return f"{dv} ~ " + " + ".join(rhs_terms)




DATA_DIR: /Users/vedikabaradwaj/Documents/pdi_pensions/regression_analysis/analytic samples/recruitment
DATA_DIR exists? True
CSV files found:
['analytic_sample_0809_recruitment.csv', 'analytic_sample_0910_recruitment.csv', 'analytic_sample_1011_recruitment.csv', 'analytic_sample_1112_recruitment.csv', 'analytic_sample_1213_recruitment.csv', 'analytic_sample_1314_recruitment.csv', 'analytic_sample_1415_recruitment.csv', 'analytic_sample_1516_recruitment.csv', 'analytic_sample_1617_recruitment.csv']


In [5]:

# -------------------------------------------------------------------
# FILE LOADING + NY REMOVAL
# -------------------------------------------------------------------

def pair_to_file(y0, y1):
    yy0 = str(y0)[-2:]
    yy1 = str(y1)[-2:]
    return DATA_DIR / FILE_PATTERN.format(yy0=yy0, yy1=yy1)


def remove_ny(df):
    before = len(df)
    mask   = pd.Series(False, index=df.index)
    used_cols = []

    # Numeric FIPS columns — use stfips_t pre-2013, stfips post-2013
    for col in ["stfips_t", "stfips"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.isin(EXCLUDED_STATE_FIPS)
            used_cols.append(col)

    # String state name columns
    for col in ["state_name_t", "state_abbrev_t", "state_t"]:
        if col in df.columns:
            vals = df[col].astype(str).str.strip().str.lower()
            mask = mask | vals.isin(EXCLUDED_STATE_NAMES) | vals.str.contains("new york", na=False)
            used_cols.append(col)

    # NYC CBSA code
    for col in ["cbsafips_t", "cbsa_t"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            mask = mask | vals.eq(35620)
            used_cols.append(col)

    df2 = df.loc[~mask].copy()
    print(f"Removed NY rows: {before - len(df2):,} using columns {sorted(set(used_cols))}")
    return df2


def load_pair(y0, y1):
    path = pair_to_file(y0, y1)
    print("\n" + "=" * 90)
    print(f"PAIR {y0}-{y1}  |  Looking for: {path.name}")

    if not path.exists():
        print(f"WARNING: file not found, skipping.")
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.lower().str.strip()
    df[PAIR_VAR] = f"{str(y0)[-2:]}{str(y1)[-2:]}"

    print(f"Loaded shape: {df.shape}")
    df = remove_ny(df)
    print(f"Shape after NY removal: {df.shape}")
    print(f"Detected weight variable: {detect_weight_var(df)}")
    return df

# -------------------------------------------------------------------
# DATA CLEANING
# -------------------------------------------------------------------

def clean_categorical_series(s):
    out = s.astype("object")
    out = out.where(~pd.isna(out), "MISSING")
    out = out.astype(str).str.strip()
    out = out.replace({
        "": "MISSING", "<NA>": "MISSING", "nan": "MISSING",
        "NaN": "MISSING", "None": "MISSING", "none": "MISSING",
    })
    return out.astype("object")


def detect_weight_var(df):
    for c in WEIGHT_CANDIDATES:
        if c in df.columns:
            return c
    return None


def detect_earnwt_var(df):
    for c in EARNWT_CANDIDATES:
        if c in df.columns:
            return c
    return None


def prepare_model_data(df, controls):
    base_weight_var = detect_weight_var(df)
    if base_weight_var is None:
        return None, None, "No weight variable found"

    missing = [c for c in [DV, TREATMENT_VAR] if c not in df.columns]
    if missing:
        return None, None, f"Missing required columns: {missing}"

    # If log earnings is a control, use earnwt_t (the CPS earnings weight)
    # rather than weight_t. earnwt_t is the correct weight for regressions
    # that include earnings as a covariate.
    use_earnings = "np.log(earnwke_t)" in controls
    earnwt_var   = detect_earnwt_var(df) if use_earnings else None
    weight_var   = earnwt_var if earnwt_var is not None else base_weight_var

    data = df.copy()

    data[weight_var]    = pd.to_numeric(data[weight_var],    errors="coerce")
    data[DV]            = pd.to_numeric(data[DV],            errors="coerce")
    data[TREATMENT_VAR] = pd.to_numeric(data[TREATMENT_VAR], errors="coerce")

    numeric_cols = [DV, TREATMENT_VAR, weight_var]
    for term in controls:
        if not is_categorical_term(term):
            numeric_cols.extend(extract_needed_columns([term]))
    numeric_cols = sorted(set(c for c in numeric_cols if c in data.columns))

    for c in numeric_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    for term in controls:
        if is_categorical_term(term):
            c = categorical_column_from_term(term)
            if c and c in data.columns:
                data[c] = clean_categorical_series(data[c])

    # Log earnings requires positive earnings.
    if use_earnings and "earnwke_t" in data.columns:
        data = data[data["earnwke_t"] > 0].copy()

    data = data.dropna(subset=numeric_cols).copy()
    data = data[data[weight_var] > 0].copy()

    if data.empty:
        return None, weight_var, "No rows left after cleaning"

    return data, weight_var, None

def classify_term(term):
    term_l = term.lower()
    if term == "Intercept":
        return "intercept"
    if term == TREATMENT_VAR:
        return "treatment"
    if "race" in term_l:
        return "race"
    if "sex" in term_l:
        return "sex"
    if any(x in term_l for x in ["hisp", "ethnic", "ethnicity"]):
        return "ethnicity_hispanic"
    if any(x in term_l for x in ["docc", "occ"]):
        return "occupation"
    if "grade" in term_l or "educ" in term_l:
        return "education"
    if "ind" in term_l or "naics" in term_l:
        return "industry"
    if "union" in term_l:
        return "union"
    if "age" in term_l:
        return "age"
    if "earn" in term_l or "log" in term_l:
        return "earnings"
    return "other"


def run_weighted_lpm_all_terms(df, y0, y1):
    controls = available_terms(df, ALL_CONTROL_CANDIDATES)
    formula  = build_formula(DV, controls)

    data, weight_var, error = prepare_model_data(df, controls)
    if error:
        print(f"Skipping {y0}-{y1}: {error}")
        return []

    print(f"\nRunning weighted full-covariate LPM for {y0}-{y1}")
    print("Formula:", formula)
    print(f"Rows used: {len(data):,} / {len(df):,}")
    print("Weight variable:", weight_var)

    try:
        result = smf.wls(
            formula=formula,
            data=data,
            weights=data[weight_var]
        ).fit(cov_type=COV_TYPE)
    except Exception as e:
        print(f"Regression failed for {y0}-{y1}: {e}")
        return []

    conf_int = result.conf_int()
    rows = []

    for term in result.params.index:
        coef     = result.params.get(term, np.nan)
        se       = result.bse.get(term, np.nan)
        pval     = result.pvalues.get(term, np.nan)
        ci_low   = conf_int.loc[term, 0] if term in conf_int.index else np.nan
        ci_high  = conf_int.loc[term, 1] if term in conf_int.index else np.nan

        rows.append({
            "pair":                   f"{y0}-{y1}",
            "pair_code":              f"{str(y0)[-2:]}{str(y1)[-2:]}",
            "dv":                     DV,
            "term":                   term,
            "term_group":             classify_term(term),
            "coef":                   coef,
            "coef_pct_points":        coef    * 100 if pd.notna(coef)    else np.nan,
            "std_err":                se,
            "std_err_pct_points":     se      * 100 if pd.notna(se)      else np.nan,
            "p_value":                pval,
            "ci_low":                 ci_low,
            "ci_high":                ci_high,
            "ci_low_pct_points":      ci_low  * 100 if pd.notna(ci_low)  else np.nan,
            "ci_high_pct_points":     ci_high * 100 if pd.notna(ci_high) else np.nan,
            "nobs":                   int(result.nobs),
            "r_squared":              result.rsquared,
            "weight_var":             weight_var,
            "controls":               ", ".join(controls),
            "file":                   pair_to_file(y0, y1).name,
        })

    return rows

# -------------------------------------------------------------------
# RUN ALL YEAR PAIRS
# -------------------------------------------------------------------

all_coef_rows = []
loaded_pairs  = {}

for y0, y1 in YEAR_PAIRS:
    df_pair = load_pair(y0, y1)
    if df_pair is None:
        continue

    loaded_pairs[f"{y0}-{y1}"] = df_pair
    rows = run_weighted_lpm_all_terms(df_pair, y0, y1)
    all_coef_rows.extend(rows)

all_results_table = pd.DataFrame(all_coef_rows)

print("\n" + "=" * 90)
print("DONE")
print(f"Coefficient rows produced: {len(all_results_table):,}")
print(f"Year pairs successfully loaded: {list(loaded_pairs.keys())}")
display(all_results_table.head(20))

# -------------------------------------------------------------------
# TABLE 1: ILLINOIS EFFECT BY YEAR PAIR
# -------------------------------------------------------------------

illinois_effect_table = (
    all_results_table
    .query("term_group == 'treatment'")
    [["pair", "pair_code", "term", "coef", "coef_pct_points", "std_err", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var", "file"]]
    .sort_values("pair_code")
    .reset_index(drop=True)
)
display(illinois_effect_table)

# -------------------------------------------------------------------
# TABLE 2: ALL COVARIATE EFFECTS
# -------------------------------------------------------------------

all_covariate_effects_table = (
    all_results_table
    .query("term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef", "coef_pct_points", "std_err",
      "std_err_pct_points", "p_value", "ci_low_pct_points", "ci_high_pct_points",
      "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(all_covariate_effects_table)

# -------------------------------------------------------------------
# TABLE 3: DEMOGRAPHIC, OCCUPATION, INDUSTRY, UNION, AGE, EARNINGS
# -------------------------------------------------------------------

focus_groups = [
    "treatment", "race", "sex", "ethnicity_hispanic",
    "occupation", "education", "industry", "union", "age", "earnings"
]

focus_effects_table = (
    all_results_table
    .query("term_group in @focus_groups and term != 'Intercept'")
    [["pair", "pair_code", "term_group", "term", "coef_pct_points", "std_err_pct_points",
      "p_value", "ci_low_pct_points", "ci_high_pct_points", "nobs", "r_squared", "weight_var"]]
    .sort_values(["pair_code", "term_group", "term"])
    .reset_index(drop=True)
)
display(focus_effects_table)

# -------------------------------------------------------------------
# TABLE 4: AVERAGE COEFFICIENT ACROSS YEAR PAIRS BY TERM
# -------------------------------------------------------------------

average_effects_by_term = (
    all_results_table
    .query("term != 'Intercept'")
    .groupby(["term_group", "term"], as_index=False)
    .agg(
        mean_coef=("coef", "mean"),
        mean_coef_pct_points=("coef_pct_points", "mean"),
        median_coef_pct_points=("coef_pct_points", "median"),
        num_year_pairs=("pair_code", "nunique"),
        mean_nobs=("nobs", "mean"),
    )
    .sort_values(["term_group", "term"])
    .reset_index(drop=True)
)
display(average_effects_by_term)

# -------------------------------------------------------------------
# SAVE
# -------------------------------------------------------------------

out_all_csv      = OUTPUT_DIR / "recruitment_weighted_all_covariate_coefficients.csv"
out_illinois_csv = OUTPUT_DIR / "recruitment_weighted_illinois_effects.csv"
out_focus_csv    = OUTPUT_DIR / "recruitment_weighted_demographic_occupation_effects.csv"
out_avg_csv      = OUTPUT_DIR / "recruitment_weighted_average_effects_by_term.csv"
out_xlsx         = OUTPUT_DIR / "recruitment_weighted_regression_tables.xlsx"

all_results_table.to_csv(out_all_csv, index=False)
illinois_effect_table.to_csv(out_illinois_csv, index=False)
focus_effects_table.to_csv(out_focus_csv, index=False)
average_effects_by_term.to_csv(out_avg_csv, index=False)

with pd.ExcelWriter(out_xlsx) as writer:
    illinois_effect_table.to_excel(writer,        sheet_name="illinois_effects",    index=False)
    all_covariate_effects_table.to_excel(writer,  sheet_name="all_covariates",      index=False)
    focus_effects_table.to_excel(writer,          sheet_name="demo_occ_effects",    index=False)
    average_effects_by_term.to_excel(writer,      sheet_name="avg_by_term",         index=False)
    all_results_table.to_excel(writer,            sheet_name="raw_all_terms",       index=False)

print("Saved:")
print("-", out_all_csv)
print("-", out_illinois_csv)
print("-", out_focus_csv)
print("-", out_avg_csv)
print("-", out_xlsx)


PAIR 2008-2009  |  Looking for: analytic_sample_0809_recruitment.csv
Loaded shape: (18302, 205)
Removed NY rows: 8,859 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (9443, 205)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2008-2009
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(unionmme_t) + C(unioncov_t)
Rows used: 4,842 / 9,443
Weight variable: earnwt_t

PAIR 2009-2010  |  Looking for: analytic_sample_0910_recruitment.csv
Loaded shape: (18724, 203)
Removed NY rows: 8,993 using columns ['cbsafips_t', 'state_t', 'stfips_t']
Shape after NY removal: (9731, 203)
Detected weight variable: weight_t

Running weighted full-covariate LPM for 2009-2010
Formula: joined_state_local ~ illinois + age_t + I(age_t**2) + np.log(earnwke_t) + C(sex_t) + C(race_t) + C(ethnic_t) + C(grade92_t) + C(docc00_t) + C(ind02_t) + C(union

,pair,pair_code,dv,term,term_group,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low,ci_high,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,controls,file
0,2008-2009,0809,joined_state_local,Intercept,intercept,0.010602,1.060242,0.028551,2.855090,0.710376,-0.045356,0.066561,-4.535631,6.656116,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
1,2008-2009,0809,joined_state_local,C(sex_t)[T.2],sex,0.012206,1.220556,0.007048,0.704757,0.083294,-0.001607,0.026019,-0.160741,2.601854,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
2,2008-2009,0809,joined_state_local,C(race_t)[T.10],race,-0.008213,-0.821294,0.038055,3.805456,0.829128,-0.082799,0.066373,-8.279852,6.637263,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
3,2008-2009,0809,joined_state_local,C(race_t)[T.13],race,-0.051435,-5.143539,0.037257,3.725739,0.167420,-0.124459,0.021588,-12.445854,2.158775,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
4,2008-2009,0809,joined_state_local,C(race_t)[T.15],race,0.000806,0.080550,0.022217,2.221703,0.971078,-0.042739,0.044350,-4.273907,4.435008,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
5,2008-2009,0809,joined_state_local,C(race_t)[T.2],race,0.009012,0.901220,0.013561,1.356135,0.506338,-0.017568,0.035592,-1.756756,3.559196,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
6,2008-2009,0809,joined_state_local,C(race_t)[T.3],race,0.065139,6.513880,0.079417,7.941736,0.412097,-0.090516,0.220794,-9.051636,22.079397,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
7,2008-2009,0809,joined_state_local,C(race_t)[T.4],race,-0.028721,-2.872082,0.014502,1.450156,0.047644,-0.057143,-0.000298,-5.714335,-0.029828,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
8,2008-2009,0809,joined_state_local,C(race_t)[T.5],race,-0.050450,-5.044962,0.045930,4.592951,0.272024,-0.140470,0.039571,-14.046980,3.957056,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv
9,2008-2009,0809,joined_state_local,C(race_t)[T.6],race,-0.006631,-0.663133,0.021570,2.157021,0.758516,-0.048908,0.035646,-4.890816,3.564551,4842,0.101551,earnwt_t,"age_t, I(age_t**2), np.log(earnwke_t), C(sex_t), C(race_t), C(ethnic_t), C(grade92_t), C(docc00_t), C(ind02_t), C(unionmme_t), C(unioncov_t)",analytic_sample_0809_recruitment.csv


,pair,pair_code,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var,file
0,2008-2009,0809,illinois,-0.009145,-0.914481,0.006191,0.619125,0.139661,-2.127944,0.298981,4842,0.101551,earnwt_t,analytic_sample_0809_recruitment.csv
1,2009-2010,0910,illinois,-0.003637,-0.363683,0.006091,0.609102,0.550454,-1.557502,0.830136,4816,0.108443,earnwt_t,analytic_sample_0910_recruitment.csv
2,2010-2011,1011,illinois,-0.003390,-0.338953,0.006268,0.626769,0.588650,-1.567398,0.889492,4716,0.085575,earnwt_t,analytic_sample_1011_recruitment.csv
3,2011-2012,1112,illinois,-0.008426,-0.842580,0.006103,0.610267,0.167379,-2.038682,0.353522,4599,0.095239,earnwt_t,analytic_sample_1112_recruitment.csv
4,2012-2013,1213,illinois,0.007554,0.755399,0.006182,0.618239,0.221762,-0.456327,1.967124,4616,0.104702,earnwt_t,analytic_sample_1213_recruitment.csv
5,2013-2014,1314,illinois,-0.002106,-0.210566,0.006006,0.600626,0.725905,-1.387772,0.966639,3960,0.100440,earnwt_t,analytic_sample_1314_recruitment.csv
6,2014-2015,1415,illinois,0.001435,0.143502,0.006439,0.643895,0.823640,-1.118508,1.405513,2955,0.190935,earnwt_t,analytic_sample_1415_recruitment.csv
7,2015-2016,1516,illinois,-0.020377,-2.037699,0.006270,0.626986,0.001154,-3.266569,-0.808828,3332,0.194397,earnwt_t,analytic_sample_1516_recruitment.csv
8,2016-2017,1617,illinois,-0.003997,-0.399654,0.006153,0.615350,0.516032,-1.605717,0.806410,3312,0.157496,earnwt_t,analytic_sample_1617_recruitment.csv


,pair,pair_code,term_group,term,coef,coef_pct_points,std_err,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.000016,-0.001565,0.000016,0.001637,0.339010,-0.004773,0.001643,4842,0.101551,earnwt_t
1,2008-2009,0809,age,age_t,0.001793,0.179288,0.001429,0.142933,0.209714,-0.100855,0.459431,4842,0.101551,earnwt_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.007725,-0.772519,0.005611,0.561070,0.168553,-1.872196,0.327157,4842,0.101551,earnwt_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-0.053073,-5.307321,0.025859,2.585938,0.040133,-10.375667,-0.238975,4842,0.101551,earnwt_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-0.059269,-5.926919,0.023047,2.304748,0.010123,-10.444143,-1.409695,4842,0.101551,earnwt_t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2640,2016-2017,1617,sex,C(sex_t)[T.2],0.008138,0.813789,0.006647,0.664690,0.220834,-0.488981,2.116558,3312,0.157496,earnwt_t
2641,2016-2017,1617,treatment,illinois,-0.003997,-0.399654,0.006153,0.615350,0.516032,-1.605717,0.806410,3312,0.157496,earnwt_t
2642,2016-2017,1617,union,C(unioncov_t)[T.2.0],-0.029629,-2.962852,0.044030,4.403005,0.501001,-11.592583,5.666880,3312,0.157496,earnwt_t
2643,2016-2017,1617,union,C(unioncov_t)[T.MISSING],-0.002352,-0.235157,0.022546,2.254648,0.916932,-4.654186,4.183872,3312,0.157496,earnwt_t


,pair,pair_code,term_group,term,coef_pct_points,std_err_pct_points,p_value,ci_low_pct_points,ci_high_pct_points,nobs,r_squared,weight_var
0,2008-2009,0809,age,I(age_t ** 2),-0.001565,0.001637,0.339010,-0.004773,0.001643,4842,0.101551,earnwt_t
1,2008-2009,0809,age,age_t,0.179288,0.142933,0.209714,-0.100855,0.459431,4842,0.101551,earnwt_t
2,2008-2009,0809,earnings,np.log(earnwke_t),-0.772519,0.561070,0.168553,-1.872196,0.327157,4842,0.101551,earnwt_t
3,2008-2009,0809,education,C(grade92_t)[T.32],-5.307321,2.585938,0.040133,-10.375667,-0.238975,4842,0.101551,earnwt_t
4,2008-2009,0809,education,C(grade92_t)[T.33],-5.926919,2.304748,0.010123,-10.444143,-1.409695,4842,0.101551,earnwt_t
...,...,...,...,...,...,...,...,...,...,...,...,...
2640,2016-2017,1617,sex,C(sex_t)[T.2],0.813789,0.664690,0.220834,-0.488981,2.116558,3312,0.157496,earnwt_t
2641,2016-2017,1617,treatment,illinois,-0.399654,0.615350,0.516032,-1.605717,0.806410,3312,0.157496,earnwt_t
2642,2016-2017,1617,union,C(unioncov_t)[T.2.0],-2.962852,4.403005,0.501001,-11.592583,5.666880,3312,0.157496,earnwt_t
2643,2016-2017,1617,union,C(unioncov_t)[T.MISSING],-0.235157,2.254648,0.916932,-4.654186,4.183872,3312,0.157496,earnwt_t


,term_group,term,mean_coef,mean_coef_pct_points,median_coef_pct_points,num_year_pairs,mean_nobs
0,age,I(age_t ** 2),-0.000003,-0.000330,-0.000283,9,4127.555556
1,age,age_t,0.000313,0.031262,0.008646,9,4127.555556
2,earnings,np.log(earnwke_t),-0.005104,-0.510382,-0.712911,9,4127.555556
3,education,C(grade92_t)[T.32],-0.000909,-0.090933,-0.080790,9,4127.555556
4,education,C(grade92_t)[T.33],0.002148,0.214752,-0.478546,9,4127.555556
...,...,...,...,...,...,...,...
329,sex,C(sex_t)[T.2],0.006737,0.673695,0.732883,9,4127.555556
330,treatment,illinois,-0.004676,-0.467635,-0.363683,9,4127.555556
331,union,C(unioncov_t)[T.2.0],-0.008502,-0.850157,-1.970456,9,4127.555556
332,union,C(unioncov_t)[T.MISSING],0.014617,1.461689,3.730810,9,4127.555556


Saved:
- /Users/vedikabaradwaj/Documents/recruitment_weighted_all_covariate_coefficients.csv
- /Users/vedikabaradwaj/Documents/recruitment_weighted_illinois_effects.csv
- /Users/vedikabaradwaj/Documents/recruitment_weighted_demographic_occupation_effects.csv
- /Users/vedikabaradwaj/Documents/recruitment_weighted_average_effects_by_term.csv
- /Users/vedikabaradwaj/Documents/recruitment_weighted_regression_tables.xlsx
